In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import math
import torch.nn as nn
import torch.nn.functional as F
from timm.models.layers import to_2tuple, trunc_normal_
from collections import OrderedDict
from torch.utils.data import DataLoader
from torchvision import transforms
import shutil
from datasets import load_dataset
from PIL import Image

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [4]:
class UniformQuantizerObserver(nn.Module):
    def __init__(self, n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8, power_of_2=False):
        super().__init__()
        self.n_bits = int(n_bits)
        self.symmetric = bool(symmetric)
        self.channel_wise = bool(channel_wise)
        self.momentum = float(momentum)
        self.ch_axis = int(ch_axis)
        self.eps = float(eps)
        self.power_of_2 = bool(power_of_2)

        if self.symmetric:
            self.qmax_sym = float(2 ** (self.n_bits - 1) - 1)
        else:
            self.qmin = 0
            self.qmax = 2 ** self.n_bits - 1
            self.denom = float(self.qmax - self.qmin)

        self.register_buffer("scale", torch.tensor(1.0, dtype=torch.float32))

    def _ensure_shape(self, sample_scale):
        if self.scale.shape != sample_scale.shape:
            with torch.no_grad():
                self.scale.resize_(sample_scale.shape)

    def calculate_qparams(self, x):
        if self.channel_wise:
            dims = [i for i in range(x.ndim) if i != self.ch_axis]
            x_min = x.amin(dim=dims, keepdim=True)
            x_max = x.amax(dim=dims, keepdim=True)
        else:
            x_min = x.min()
            x_max = x.max()

        if self.symmetric:
            x_absmax = torch.maximum(x_min.abs(), x_max.abs())
            scale = (x_absmax / self.qmax_sym).clamp(min=self.eps)
        else:
            scale = ((x_max - x_min) / self.denom).clamp(min=self.eps)

        return scale

    @torch.no_grad()
    def init_from_tensor(self, x):
        scale = self.calculate_qparams(x)
        self._ensure_shape(scale)
        self.scale.copy_(scale.detach().to(self.scale.device))

    @torch.no_grad()
    def forward(self, x):
        if self.channel_wise and self.scale.ndim == 0:
            self.init_from_tensor(x)
            return x
        else:
            scale_b = self.calculate_qparams(x)
            s_new = scale_b.detach().to(self.scale.device)
            self.scale.lerp_(s_new, self.momentum)
            return x

    @torch.no_grad()
    def finalize_power_of_2_scale(self):
        if self.power_of_2:
            continuous_scale = self.scale

            scale_log2 = torch.log2(continuous_scale.clamp(min=self.eps))
            exponent_final = torch.round(scale_log2)
            scale_final = torch.pow(2.0, exponent_final)

            self.scale.copy_(scale_final)

In [5]:
def quantize(tensor: torch.Tensor, scale: torch.Tensor, is_channel_wise: bool, is_bias: bool = False) -> torch.Tensor:

    if is_bias:
        Q_MIN = -2**31
        Q_MAX = 2**31 - 1
        return_as_param = True
    elif is_channel_wise:
        Q_MIN = -127.0
        Q_MAX = 127.0
        return_as_param = True
    else:
        Q_MIN = -127.0
        Q_MAX = 127.0
        return_as_param = False

    if is_bias:
        scale_view = scale.flatten()
    elif is_channel_wise:
        scale_view = scale.view(-1, *[1] * (tensor.ndim - 1))
    else:
        scale_view = scale.squeeze()

    quantized_tensor = tensor / scale_view

    quantized_tensor = torch.round(quantized_tensor)

    quantized_tensor = torch.clamp(quantized_tensor, Q_MIN, Q_MAX)

    if return_as_param:
        return nn.Parameter(quantized_tensor)
    else:
        return quantized_tensor


def dequantize(quantized_tensor_float: torch.Tensor, scale: torch.Tensor, is_channel_wise: bool) -> torch.Tensor:

    if is_channel_wise:
        scale_view = scale.view(-1, *[1] * (quantized_tensor_float.ndim - 1))
    else:
        scale_view = scale.squeeze()

    dequantized_tensor = quantized_tensor_float * scale_view

    return dequantized_tensor

In [6]:
def softmax(x, dim=-1):

    max_x = torch.max(x, dim=dim, keepdim=True)[0]

    two_x = torch.pow(2.0, x - max_x)

    sum_two_x = torch.sum(two_x, dim=dim, keepdim=True)

    return two_x / sum_two_x

In [7]:
class FloatAttnMaskSoftmax(nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        self.softmax = nn.Softmax(dim=-1)
        self.num_heads = num_heads

    def forward(self, attn_float, mask, B_, N):
        attn = torch.round(attn_float)
        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
        # base-2 softmax via ln(2)
        attn = softmax(attn)
        return attn

In [142]:
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None,
                 out_features=None, act_layer=nn.ReLU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features

        self.drop = nn.Dropout(drop)

        self.fuse_linear_bn1_weight_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=True, momentum=0.1, ch_axis=0, eps=1e-8)
        self.fuse_linear_bn2_weight_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=True, momentum=0.1, ch_axis=0, eps=1e-8)

        self.fuse_linear_bn1_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)
        self.fuse_linear_bn2_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)

        self.fuse_linear_bn1 = nn.Linear(in_features, hidden_features)
        self.fuse_linear_bn2 = nn.Linear(hidden_features, out_features)

        self.register_buffer("fuse_linear_bn1_output_scale", None)
        self.register_buffer("fuse_linear_bn2_output_scale", None)

        self.is_not_quantized = True

    def forward(self, x, scale):

        if self.is_not_quantized:
            M = self.fuse_linear_bn1_output_observer.scale / (self.fuse_linear_bn1_weight_observer.scale * scale)
            M_po2 = torch.pow(2.0, torch.floor(torch.log2(M)))
            self.fuse_linear_bn1_output_scale = M_po2
            self.fuse_linear_bn1.weight = quantize(self.fuse_linear_bn1.weight, self.fuse_linear_bn1_output_observer.scale / (M_po2 * scale), is_channel_wise=True)
            self.fuse_linear_bn1.bias = quantize(self.fuse_linear_bn1.bias, self.fuse_linear_bn1_output_observer.scale / M_po2, is_channel_wise=True, is_bias=True)
            # self.fuse_linear_bn1_ReLU[0].weight = quantize(self.fuse_linear_bn1_ReLU[0].weight, self.fuse_linear_bn1_ReLU_weight_observer.scale, is_channel_wise=True)
            # self.fuse_linear_bn1_ReLU[0].bias = quantize(self.fuse_linear_bn1_ReLU[0].bias, self.fuse_linear_bn1_ReLU_weight_observer.scale * scale, is_channel_wise=True, is_bias=True)

        x = self.fuse_linear_bn1(x)
        # x = quantize(x, self.fuse_linear_bn1_ReLU_output_observer.scale / (self.fuse_linear_bn1_ReLU_weight_observer.scale * scale), is_channel_wise=False)
        x = quantize(x, self.fuse_linear_bn1_output_scale, is_channel_wise=False)

        x = nn.ReLU()(x)

        if self.is_not_quantized:
            M = scale / (self.fuse_linear_bn2_weight_observer.scale * self.fuse_linear_bn1_output_observer.scale)
            M_po2 = torch.pow(2.0, torch.floor(torch.log2(M)))
            self.fuse_linear_bn2_output_scale = M_po2
            self.fuse_linear_bn2.weight = quantize(self.fuse_linear_bn2.weight, scale / (M_po2 * self.fuse_linear_bn1_output_observer.scale), is_channel_wise=True)
            self.fuse_linear_bn2.bias = quantize(self.fuse_linear_bn2.bias, scale / M_po2, is_channel_wise=True, is_bias=True)
            # self.fuse_linear_bn2.weight = quantize(self.fuse_linear_bn2.weight, self.fuse_linear_bn2_weight_observer.scale, is_channel_wise=True)
            # self.fuse_linear_bn2.bias = quantize(self.fuse_linear_bn2.bias, self.fuse_linear_bn2_weight_observer.scale * self.fuse_linear_bn1_ReLU_output_observer.scale, is_channel_wise=True, is_bias=True)

        x = self.fuse_linear_bn2(x)
        # x = quantize(x, self.fuse_linear_bn2_output_observer.scale / (self.fuse_linear_bn2_weight_observer.scale * self.fuse_linear_bn1_ReLU_output_observer.scale), is_channel_wise=False)
        x = quantize(x, self.fuse_linear_bn2_output_scale, is_channel_wise=False)

        self.is_not_quantized = False

        return x

def window_partition(x, window_size):
    """
    Args:
        x: (B, H, W, C)
        window_size (int): window size

    Returns:
        windows: (num_windows*B, window_size, window_size, C)
    """
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    """
    Args:
        windows: (num_windows*B, window_size, window_size, C)
        window_size (int): Window size
        H (int): Height of image
        W (int): Width of image

    Returns:
        x: (B, H, W, C)
    """
    B = windows.shape[0] // (H * W // window_size // window_size)
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


class WindowAttention(nn.Module):
    r""" Window based multi-head self attention (W-MSA) module with relative position bias.
    It supports both of shifted and non-shifted window.

    Args:
        dim (int): Number of input channels.
        window_size (tuple[int]): The height and width of the window.
        num_heads (int): Number of attention heads.
        qkv_bias (bool, optional):  If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set
        attn_drop (float, optional): Dropout ratio of attention weight. Default: 0.0
        proj_drop (float, optional): Dropout ratio of output. Default: 0.0
    """

    def __init__(self, dim, window_size, num_heads, qkv_bias=True, qk_scale=None, attn_drop=0., proj_drop=0.):

        super().__init__()
        self.dim = dim
        self.window_size = window_size  # Wh, Ww
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = (0.5 if num_heads == 3 else
                     0.5 if num_heads == 6 else
                     0.25 if num_heads == 12 else
                     0.25 if num_heads == 24 else
                     head_dim ** -0.5)

        self.scale = torch.tensor(self.scale)

        # define a parameter table of relative position bias
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))  # 2*Wh-1 * 2*Ww-1, nH

        # get pair-wise relative position index for each token inside the window
        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w]))  # 2, Wh, Ww
        coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 2, Wh*Ww, Wh*Ww
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # Wh*Ww, Wh*Ww, 2
        relative_coords[:, :, 0] += self.window_size[0] - 1  # shift to start from 0
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)  # Wh*Ww, Wh*Ww
        self.register_buffer("relative_position_index", relative_position_index, persistent=False)

        self.fuse_qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        trunc_normal_(self.relative_position_bias_table, std=.02)

        self.float_mask_softmax = FloatAttnMaskSoftmax(num_heads)

        #self.softmax_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)

        self.fuse_qkv_weight_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=True, momentum=0.1, ch_axis=0, eps=1e-8)
        self.proj_weight_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=True, momentum=0.1, ch_axis=0, eps=1e-8)

        self.fuse_qkv_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)
        self.proj_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)
        self.qk_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)
        self.attn_v_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)

        self.register_buffer("fuse_qkv_output_scale", None)
        self.register_buffer("qk_output_scale", None)
        self.register_buffer("relative_position_bias", None)
        # self.register_buffer("softmax_input_scale", None)
        # self.register_buffer("softmax_output_scale", None)
        self.register_buffer("attn_v_output_scale", None)
        self.register_buffer("proj_output_scale", None)

        self.is_not_quantized = True

    def forward(self, x, scale, mask=None):
        """
        Args:
            x: input features with shape of (num_windows*B, N, C)
            mask: (0/-inf) mask with shape of (num_windows, Wh*Ww, Wh*Ww) or None
        """
        B_, N, C = x.shape

        if self.is_not_quantized:
            M = self.fuse_qkv_output_observer.scale / (self.fuse_qkv_weight_observer.scale * scale)
            M_po2 = torch.pow(2.0, torch.floor(torch.log2(M)))
            self.fuse_qkv_output_scale = M_po2
            self.fuse_qkv.weight = quantize(self.fuse_qkv.weight, self.fuse_qkv_output_observer.scale / (M_po2 * scale), is_channel_wise=True)
            self.fuse_qkv.bias = quantize(self.fuse_qkv.bias, self.fuse_qkv_output_observer.scale / M_po2, is_channel_wise=True, is_bias=True)
            # self.qkv.weight = quantize(self.qkv.weight, self.qkv_weight_observer.scale, is_channel_wise=True)
            # self.qkv.bias = quantize(self.qkv.bias, self.qkv_weight_observer.scale * scale, is_channel_wise=True, is_bias=True)

        qkv = self.fuse_qkv(x)
        # qkv = quantize(qkv, self.qkv_output_observer.scale / (self.qkv_weight_observer.scale * scale), is_channel_wise=False)
        qkv = quantize(qkv, self.fuse_qkv_output_scale, is_channel_wise=False)

        qkv = qkv.reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # make torchscript happy (cannot use tensor as tuple)

        q = quantize(q, 1/self.scale, is_channel_wise=False, is_bias=False)
        attn = torch.matmul(q, k.transpose(-2, -1))

        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1)  # Wh*Ww,Wh*Ww,nH
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()  # nH, Wh*Ww, Wh*Ww
        relative_position_bias = quantize(relative_position_bias, self.fuse_qkv_output_observer.scale ** 2, is_channel_wise=False, is_bias=True)
        self.relative_position_bias = relative_position_bias.unsqueeze(0)
        attn = attn + self.relative_position_bias
        #attn = torch.clamp(attn, -127, 127)

        self.qk_output_scale = 1 / (self.fuse_qkv_output_observer.scale ** 2)
        attn = quantize(attn, self.qk_output_scale, is_channel_wise=False)


        # self.softmax_input_scale = self.qk_output_observer.scale
        # attn = dequantize(attn, self.softmax_input_scale, is_channel_wise=False)

        attn = self.float_mask_softmax(attn, mask, B_, N)

        power = torch.log2(attn)
        power = torch.round(power)
        power = torch.clamp(power, -127, 127)
        power = power + 7
        attn = 2 ** power
        attn = torch.clamp(attn, -127, 127)

        #self.softmax_output_scale = self.softmax_output_observer.scale
        # attn = quantize(attn, torch.tensor(2.0 ** -7), is_channel_wise=False)

        x = torch.matmul(attn, v)
        self.attn_v_output_scale = self.attn_v_output_observer.scale / (2.0 ** -7 * self.fuse_qkv_output_observer.scale)
        x = quantize(x, self.attn_v_output_scale, is_channel_wise=False)

        x = x.transpose(1, 2).reshape(B_, N, C)

        if self.is_not_quantized:
            M = scale / (self.proj_weight_observer.scale * self.attn_v_output_observer.scale)
            M_po2 = torch.pow(2.0, torch.floor(torch.log2(M)))
            self.proj_output_scale = M_po2
            self.proj.weight = quantize(self.proj.weight, scale / (M_po2 * self.attn_v_output_observer.scale), is_channel_wise=True)
            self.proj.bias = quantize(self.proj.bias, scale / M_po2, is_channel_wise=True, is_bias=True)
            # self.proj.weight = quantize(self.proj.weight, self.proj_weight_observer.scale, is_channel_wise=True)
            # self.proj.bias = quantize(self.proj.bias, self.proj_weight_observer.scale * self.attn_v_output_observer.scale, is_channel_wise=True, is_bias=True)

        x = self.proj(x)
        # x = quantize(x, self.proj_output_observer.scale / (self.proj_weight_observer.scale * self.attn_v_output_observer.scale), is_channel_wise=False)
        x = quantize(x, self.proj_output_scale, is_channel_wise=False)

        self.is_not_quantized = False

        return x

class SwinTransformerBlock(nn.Module):
    r""" Swin Transformer Block.

    Args:
        dim (int): Number of input channels.
        input_resolution (tuple[int]): Input resulotion.
        num_heads (int): Number of attention heads.
        window_size (int): Window size.
        shift_size (int): Shift size for SW-MSA.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.
        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set.
        drop (float, optional): Dropout rate. Default: 0.0
        attn_drop (float, optional): Attention dropout rate. Default: 0.0
        drop_path (float, optional): Stochastic depth rate. Default: 0.0
        act_layer (nn.Module, optional): Activation layer. Default: nn.GELU
        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.LayerNorm
        fused_window_process (bool, optional): If True, use one kernel to fused window shift & window partition for acceleration, similar for the reversed part. Default: False
    """

    def __init__(self, dim, input_resolution, num_heads, window_size=7, shift_size=0,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0., drop_path=0.,
                 act_layer=nn.ReLU, fused_window_process=False):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        if min(self.input_resolution) <= self.window_size:
            # if window size is larger than input resolution, we don't partition windows
            self.shift_size = 0
            self.window_size = min(self.input_resolution)
        assert 0 <= self.shift_size < self.window_size, "shift_size must in 0-window_size"

        self.attn = WindowAttention(
            dim, window_size=to_2tuple(self.window_size), num_heads=num_heads,
            qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)

        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)

        if self.shift_size > 0:
            # calculate attention mask for SW-MSA
            H, W = self.input_resolution
            img_mask = torch.zeros((1, H, W, 1))  # 1 H W 1
            h_slices = (slice(0, -self.window_size),
                        slice(-self.window_size, -self.shift_size),
                        slice(-self.shift_size, None))
            w_slices = (slice(0, -self.window_size),
                        slice(-self.window_size, -self.shift_size),
                        slice(-self.shift_size, None))
            cnt = 0
            for h in h_slices:
                for w in w_slices:
                    img_mask[:, h, w, :] = cnt
                    cnt += 1

            mask_windows = window_partition(img_mask, self.window_size)  # nW, window_size, window_size, 1
            mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
        else:
            attn_mask = None

        self.register_buffer("attn_mask", attn_mask)
        self.fused_window_process = fused_window_process

        #self.input_norm1 = nn.Linear(dim, dim)
        #self.input_norm2 = nn.Linear(dim, dim)

        #self.input_norm1_weight_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=True, momentum=0.1, ch_axis=0, eps=1e-8)
        #self.input_norm2_weight_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=True, momentum=0.1, ch_axis=0, eps=1e-8)

        #self.input_norm1_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)
        #self.input_norm2_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)

        self.add1_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)
        self.add2_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)

        #self.register_buffer("input_norm1_output_scale", None)
        # self.register_buffer("shortcut_add1_scale", None)
        # self.register_buffer("x_add1_scale", None)
        #self.register_buffer("input_norm2_output_scale", None)
        # self.register_buffer("x_add2_scale", None)
        # self.register_buffer("x_mlp_add2_scale", None)

        self.is_not_quantized = True

    def forward(self, x, scale):
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"

        shortcut = x

        # if self.is_not_quantized:
        #     M = self.input_norm1_output_observer.scale / (self.input_norm1_weight_observer.scale * scale)
        #     M_po2 = torch.pow(2.0, torch.floor(torch.log2(M)))
        #     self.input_norm1_output_scale = M_po2
        #     self.input_norm1.weight = quantize(self.input_norm1.weight, self.input_norm1_output_observer.scale / (M_po2 * scale), is_channel_wise=True)
        #     self.input_norm1.bias = quantize(self.input_norm1.bias, self.input_norm1_output_observer.scale / M_po2, is_channel_wise=True, is_bias=True)
            # self.input_norm1.weight = quantize(self.input_norm1.weight, self.input_norm1_weight_observer.scale, is_channel_wise=True)
            # self.input_norm1.bias = quantize(self.input_norm1.bias, self.input_norm1_weight_observer.scale * scale, is_channel_wise=True, is_bias=True)

        # x = self.input_norm1(x)
        # x = quantize(x, self.input_norm1_output_observer.scale / (self.input_norm1_weight_observer.scale * scale), is_channel_wise=False)
        # x = quantize(x, self.input_norm1_output_scale, is_channel_wise=False)

        x = x.view(B, H, W, C)

        # cyclic shift
        if self.shift_size > 0:
            if not self.fused_window_process:
                shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
                # partition windows
                x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C
        else:
            shifted_x = x
            # partition windows
            x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C

        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)  # nW*B, window_size*window_size, C

        # W-MSA/SW-MSA
        attn_windows = self.attn(x_windows, scale, mask=self.attn_mask)  # nW*B, window_size*window_size, C

        # merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)

        # reverse cyclic shift
        if self.shift_size > 0:
            if not self.fused_window_process:
                shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
                x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
            x = shifted_x

        x = x.view(B, H * W, C)
        # self.shortcut_add1_scale = self.add1_output_observer.scale / scale
        # shortcut = quantize(shortcut, self.shortcut_add1_scale, is_channel_wise=False)
        # self.x_add1_scale = self.add1_output_observer.scale / attn_scale
        # x = quantize(x, self.x_add1_scale, is_channel_wise=False)
        x = shortcut + x
        x = torch.clamp(x, -127.0, 127.0)

        # FFN
        # if self.is_not_quantized:#################################
        #     M = self.input_norm2_output_observer.scale / (self.input_norm2_weight_observer.scale * self.add1_output_observer.scale)
        #     M_po2 = torch.pow(2.0, torch.floor(torch.log2(M)))
        #     self.input_norm2_output_scale = M_po2
        #     self.input_norm2.weight = quantize(self.input_norm2.weight, self.input_norm2_output_observer.scale / (M_po2 * self.add1_output_observer.scale), is_channel_wise=True)
        #     self.input_norm2.bias = quantize(self.input_norm2.bias, self.input_norm2_output_observer.scale / M_po2, is_channel_wise=True, is_bias=True)
            # self.input_norm2.weight = quantize(self.input_norm2.weight, self.input_norm2_weight_observer.scale, is_channel_wise=True)
            # self.input_norm2.bias = quantize(self.input_norm2.bias, self.input_norm2_weight_observer.scale * self.add1_output_observer.scale, is_channel_wise=True, is_bias=True)


        # x = self.input_norm2(x)
        # # x = quantize(x, self.input_norm2_output_observer.scale / (self.input_norm2_weight_observer.scale * self.add1_output_observer.scale), is_channel_wise=False)
        # x = quantize(x, self.input_norm2_output_scale, is_channel_wise=False)

        x_mlp = self.mlp(x, scale)

        # self.x_add2_scale = self.add2_output_observer.scale / self.add1_output_observer.scale
        # x = quantize(x, self.x_add2_scale, is_channel_wise=False)
        # self.x_mlp_add2_scale = self.add2_output_observer.scale / scale_mlp
        # x_mlp = quantize(x_mlp, self.x_mlp_add2_scale, is_channel_wise=False)
        x = x + x_mlp
        x = torch.clamp(x, -127.0, 127.0)

        self.is_not_quantized = False

        return x, scale


class PatchMerging(nn.Module):
    r""" Patch Merging Layer.

    Args:
        input_resolution (tuple[int]): Resolution of input feature.
        dim (int): Number of input channels.
        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.LayerNorm
    """

    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim

        self.input_norm = nn.Linear(4 * dim, 2 * dim, bias=True)
        #self.input_norm = nn.Linear(4 * dim, 4 * dim)

        #self.reduction_weight_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=True, momentum=0.1, ch_axis=0, eps=1e-8)
        self.input_norm_weight_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=True, momentum=0.1, ch_axis=0, eps=1e-8)

        #self.reduction_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)
        self.input_norm_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)

        #self.register_buffer("reduction_output_scale", None)
        self.register_buffer("input_norm_output_scale", None)

        self.is_not_quantized = True

    def forward(self, x, scale):
        """
        x: B, H*W, C
        """
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"
        assert H % 2 == 0 and W % 2 == 0, f"x size ({H}*{W}) are not even."


        x = x.view(B, H, W, C)

        x0 = x[:, 0::2, 0::2, :]  # B H/2 W/2 C
        x1 = x[:, 1::2, 0::2, :]  # B H/2 W/2 C
        x2 = x[:, 0::2, 1::2, :]  # B H/2 W/2 C
        x3 = x[:, 1::2, 1::2, :]  # B H/2 W/2 C
        x = torch.cat([x0, x1, x2, x3], -1)  # B H/2 W/2 4*C
        x = x.view(B, -1, 4 * C)  # B H/2*W/2 4*C

        if self.is_not_quantized:###################################
            M = self.input_norm_output_observer.scale / (self.input_norm_weight_observer.scale * scale)
            M_po2 = torch.pow(2.0, torch.floor(torch.log2(M)))
            self.input_norm_output_scale = M_po2
            self.input_norm.weight = quantize(self.input_norm.weight, self.input_norm_output_observer.scale / (M_po2 * scale), is_channel_wise=True)
            self.input_norm.bias = quantize(self.input_norm.bias, self.input_norm_output_observer.scale / M_po2, is_channel_wise=True, is_bias=True)
            # self.input_norm.weight = quantize(self.input_norm.weight, self.input_norm_weight_observer.scale, is_channel_wise=True)
            # self.input_norm.bias = quantize(self.input_norm.bias, self.input_norm_weight_observer.scale * scale, is_channel_wise=True, is_bias=True)


        x = self.input_norm(x)
        # x = quantize(x, self.input_norm_output_observer.scale / (self.input_norm_weight_observer.scale * scale), is_channel_wise=False)
        x = quantize(x, self.input_norm_output_scale, is_channel_wise=False)

        # if self.is_not_quantized:
        #     M = self.reduction_output_observer.scale / (self.reduction_weight_observer.scale * self.input_norm_output_observer.scale)
        #     M_po2 = torch.pow(2.0, torch.floor(torch.log2(M)))
        #     self.reduction_output_scale = M_po2
        #     self.reduction.weight = quantize(self.reduction.weight, self.reduction_output_observer.scale / (M_po2 * self.input_norm_output_observer.scale), is_channel_wise=True)
            # self.reduction.weight = quantize(self.reduction.weight, self.reduction_weight_observer.scale, is_channel_wise=True)

        # x = self.reduction(x)
        # # x = quantize(x, self.reduction_output_observer.scale / (self.reduction_weight_observer.scale * self.input_norm_output_observer.scale), is_channel_wise=False)
        # x = quantize(x, self.reduction_output_scale, is_channel_wise=False)

        self.is_not_quantized = False

        return x, self.input_norm_output_observer.scale


class BasicLayer(nn.Module):
    """ A basic Swin Transformer layer for one stage.

    Args:
        dim (int): Number of input channels.
        input_resolution (tuple[int]): Input resolution.
        depth (int): Number of blocks.
        num_heads (int): Number of attention heads.
        window_size (int): Local window size.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.
        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set.
        drop (float, optional): Dropout rate. Default: 0.0
        attn_drop (float, optional): Attention dropout rate. Default: 0.0
        drop_path (float | tuple[float], optional): Stochastic depth rate. Default: 0.0
        norm_layer (nn.Module, optional): Normalization layer. Default: nn.LayerNorm
        downsample (nn.Module | None, optional): Downsample layer at the end of the layer. Default: None
        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False.
        fused_window_process (bool, optional): If True, use one kernel to fused window shift & window partition for acceleration, similar for the reversed part. Default: False
    """

    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., downsample=None, use_checkpoint=False,
                 fused_window_process=False):

        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.depth = depth
        self.use_checkpoint = use_checkpoint

        # build blocks
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(dim=dim, input_resolution=input_resolution,
                                 num_heads=num_heads, window_size=window_size,
                                 shift_size=0 if (i % 2 == 0) else window_size // 2,
                                 mlp_ratio=mlp_ratio,
                                 qkv_bias=qkv_bias, qk_scale=qk_scale,
                                 drop=drop, attn_drop=attn_drop,
                                 drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                                 fused_window_process=fused_window_process)
            for i in range(depth)])

        # patch merging layer
        if downsample is not None:
            self.downsample = downsample(input_resolution, dim=dim)
        else:
            self.downsample = None

    def forward(self, x, scale):
        x, current_scale = self.blocks[0](x, scale)
        for blk in self.blocks[1:]:
            x, current_scale = blk(x, current_scale)
        if self.downsample is not None:
            x, current_scale = self.downsample(x, current_scale)
        return x, current_scale


class PatchEmbed(nn.Module):
    r""" Image to Patch Embedding

    Args:
        img_size (int): Image size.  Default: 224.
        patch_size (int): Patch token size. Default: 4.
        in_chans (int): Number of input image channels. Default: 3.
        embed_dim (int): Number of linear projection output channels. Default: 96.
        norm_layer (nn.Module, optional): Normalization layer. Default: None
    """

    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        patches_resolution = [img_size[0] // patch_size[0], img_size[1] // patch_size[1]]
        self.img_size = img_size
        self.patch_size = patch_size
        self.patches_resolution = patches_resolution
        self.num_patches = patches_resolution[0] * patches_resolution[1]

        self.in_chans = in_chans
        self.embed_dim = embed_dim

        self.fused_proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

        self.fused_proj_weight_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=True, momentum=0.1, ch_axis=0, eps=1e-8)

        self.fused_proj_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)

        self.register_buffer("fused_proj_output_scale", None)

        self.is_not_quantized = True

    def forward(self, x, scale):
        B, C, H, W = x.shape
        # FIXME look at relaxing size constraints
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."

        if self.is_not_quantized:
            M = self.fused_proj_output_observer.scale / (self.fused_proj_weight_observer.scale * scale)
            M_po2 = torch.pow(2.0, torch.floor(torch.log2(M)))
            self.fused_proj_output_scale = M_po2
            self.fused_proj.weight = quantize(self.fused_proj.weight, self.fused_proj_output_observer.scale / (M_po2 * scale), is_channel_wise=True)
            self.fused_proj.bias = quantize(self.fused_proj.bias, self.fused_proj_output_observer.scale / M_po2, is_channel_wise=False, is_bias=True)
            # self.fused_proj.weight = quantize(self.fused_proj.weight, self.fused_proj_weight_observer.scale, is_channel_wise=True)
            # self.fused_proj.bias = quantize(self.fused_proj.bias, self.fused_proj_weight_observer.scale * scale, is_channel_wise=False, is_bias=True)


        x = self.fused_proj(x)

        x = x.flatten(2).transpose(1, 2)
        # x = quantize(x, self.fused_proj_output_observer.scale / (self.fused_proj_weight_observer.scale * scale), is_channel_wise=False)
        x = quantize(x, self.fused_proj_output_scale, is_channel_wise=False)

        self.is_not_quantized = False

        return x, self.fused_proj_output_observer.scale


class SwinTransformer(nn.Module):
    r""" Swin Transformer
        A PyTorch impl of : `Swin Transformer: Hierarchical Vision Transformer using Shifted Windows`  -
          https://arxiv.org/pdf/2103.14030

    Args:
        img_size (int | tuple(int)): Input image size. Default 224
        patch_size (int | tuple(int)): Patch size. Default: 4
        in_chans (int): Number of input image channels. Default: 3
        num_classes (int): Number of classes for classification head. Default: 1000
        embed_dim (int): Patch embedding dimension. Default: 96
        depths (tuple(int)): Depth of each Swin Transformer layer.
        num_heads (tuple(int)): Number of attention heads in different layers.
        window_size (int): Window size. Default: 7
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim. Default: 4
        qkv_bias (bool): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float): Override default qk scale of head_dim ** -0.5 if set. Default: None
        drop_rate (float): Dropout rate. Default: 0
        attn_drop_rate (float): Attention dropout rate. Default: 0
        drop_path_rate (float): Stochastic depth rate. Default: 0.1
        norm_layer (nn.Module): Normalization layer. Default: nn.LayerNorm.
        ape (bool): If True, add absolute position embedding to the patch embedding. Default: False
        patch_norm (bool): If True, add normalization after patch embedding. Default: True
        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False
        fused_window_process (bool, optional): If True, use one kernel to fused window shift & window partition for acceleration, similar for the reversed part. Default: False
    """

    def __init__(self, img_size=224, patch_size=4, in_chans=3, num_classes=1000,
                 embed_dim=96, depths=[2, 2, 6, 2], num_heads=[3, 6, 12, 24],
                 window_size=7, mlp_ratio=4., qkv_bias=True, qk_scale=None,
                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1,
                 ape=False, patch_norm=True,
                 use_checkpoint=False, fused_window_process=False, **kwargs):
        super().__init__()

        self.num_classes = num_classes
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.ape = ape
        self.patch_norm = patch_norm
        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))
        self.mlp_ratio = mlp_ratio

        # split image into non-overlapping patches
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)
        num_patches = self.patch_embed.num_patches
        patches_resolution = self.patch_embed.patches_resolution
        self.patches_resolution = patches_resolution

        # absolute position embedding
        if self.ape:
            self.absolute_pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
            trunc_normal_(self.absolute_pos_embed, std=.02)

        # stochastic depth
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]  # stochastic depth decay rule

        # build layers
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            layer = BasicLayer(dim=int(embed_dim * 2 ** i_layer),
                               input_resolution=(patches_resolution[0] // (2 ** i_layer),
                                                 patches_resolution[1] // (2 ** i_layer)),
                               depth=depths[i_layer],
                               num_heads=num_heads[i_layer],
                               window_size=window_size,
                               mlp_ratio=self.mlp_ratio,
                               qkv_bias=qkv_bias, qk_scale=qk_scale,
                               drop=drop_rate, attn_drop=attn_drop_rate,
                               drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],
                               downsample=PatchMerging if (i_layer < self.num_layers - 1) else None,
                               use_checkpoint=use_checkpoint,
                               fused_window_process=fused_window_process)
            self.layers.append(layer)

        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(OrderedDict([("fc", nn.Linear(self.num_features, num_classes))])) if num_classes > 0 else nn.Identity()

        self.head_weight_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=True, momentum=0.1, ch_axis=0, eps=1e-8)

        self.input_norm = nn.Linear(self.num_features, self.num_features)

        self.input_norm_weight_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=True, momentum=0.1, ch_axis=0, eps=1e-8)
        self.input_norm_output_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)

        self.input_observer = UniformQuantizerObserver(n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8)

        self.register_buffer("input_scale", None)
        self.register_buffer("input_norm_output_scale", None)
        self.register_buffer("output_scale", None)

        self.is_not_quantized = True

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'absolute_pos_embed'}

    @torch.jit.ignore
    def no_weight_decay_keywords(self):
        return {'relative_position_bias_table'}

    def forward_features(self, x):
        self.input_scale = self.input_observer.scale
        x = quantize(x, self.input_scale, is_channel_wise=False)
        x, scale = self.patch_embed(x, self.input_observer.scale)
        if self.ape:
            x = x + self.absolute_pos_embed

        x, current_scale = self.layers[0](x, scale)

        for layer in self.layers[1:]:
            x, current_scale = layer(x, current_scale)

        if self.is_not_quantized:
            M = self.input_norm_output_observer.scale / (self.input_norm_weight_observer.scale * current_scale)
            M_po2 = torch.pow(2.0, torch.floor(torch.log2(M)))
            self.input_norm_output_scale = M_po2
            self.input_norm.weight = quantize(self.input_norm.weight, self.input_norm_output_observer.scale / (M_po2 * current_scale), is_channel_wise=True)
            self.input_norm.bias = quantize(self.input_norm.bias, self.input_norm_output_observer.scale / M_po2, is_channel_wise=True, is_bias=True)
            # self.input_norm.weight = quantize(self.input_norm.weight, self.input_norm_weight_observer.scale, is_channel_wise=True)
            # self.input_norm.bias = quantize(self.input_norm.bias, self.input_norm_weight_observer.scale * current_scale, is_channel_wise=True, is_bias=True)


        x = self.input_norm(x)
        # x = quantize(x, self.input_norm_output_observer.scale / (self.input_norm_weight_observer.scale * current_scale), is_channel_wise=False)
        x = quantize(x, self.input_norm_output_scale, is_channel_wise=False)

        x = self.avgpool(x.transpose(1, 2))  # B C 1
        x = torch.flatten(x, 1)
        return x

    def forward(self, x):
        x = self.forward_features(x)
        if self.is_not_quantized:
            self.head[0].weight = quantize(self.head[0].weight, self.head_weight_observer.scale, is_channel_wise=True)
            self.head[0].bias = quantize(self.head[0].bias, self.head_weight_observer.scale * self.input_norm_output_observer.scale, is_channel_wise=True, is_bias=True)

        x = self.head(x)
        self.output_scale = self.head_weight_observer.scale * self.input_norm_output_observer.scale
        x = dequantize(x, self.output_scale, is_channel_wise=False)

        self.is_not_quantized = False

        return x

In [148]:
# 2. Create your custom implementation
custom_model = SwinTransformer(
    img_size=224,
    patch_size=4,
    in_chans=3,
    num_classes=1000,
    embed_dim=96,
    depths=[2, 2, 6, 2],
    num_heads=[3, 6, 12, 24],
    ape=False,
    patch_norm=True,
    use_checkpoint=False,
    fused_window_process=False,
)

In [10]:
orig_state = torch.load("/content/drive/MyDrive/BN+softmax+ReLU+fusedBN+foldedBN+fusedReLU+PTQ_Observation7.pth")

In [149]:
for name, param in orig_state.items():
    # Check if the name corresponds to an observer's scale
    if ".scale" in name and "observer" in name:
        # Construct the path to the observer module
        parts = name.split('.')
        current_module = custom_model
        module_found = True
        for i, part in enumerate(parts[:-1]):
            if hasattr(current_module, part):
                current_module = getattr(current_module, part)
            else:
                # If a part of the path doesn't exist, this key is not applicable
                module_found = False
                break

        if module_found:
            attr_name = parts[-1] # This should be 'scale'
            if hasattr(current_module, attr_name):
                # Directly assign the parameter from the loaded state to the model's buffer
                # This will resize the buffer if necessary, fixing the shape mismatch.
                with torch.no_grad(): # Ensure this operation doesn't track gradients
                    # Ensure device consistency by moving param to buffer's device if different
                    if current_module.scale.device != param.device:
                        setattr(current_module, attr_name, param.to(current_module.scale.device))
                    else:
                        setattr(current_module, attr_name, param)
            else:
                print(f"Warning: Attribute '{attr_name}' not found in module '{'.'.join(parts[:-1])}' for key '{name}'. Skipping.")
        else:
            print(f"Warning: Module path for key '{name}' not fully resolvable. Skipping.")


# Load into your model
missing, unexpected = custom_model.load_state_dict(orig_state, strict=False)


# Optionally inspect for any mismatches
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

Missing keys: []
Unexpected keys: []


In [12]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: 
Token is valid (permission: fineGrained).
The token `swin` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `swin`


In [150]:
# ----------------------------------------------------------------------------
# 1) Load ImageNet-1k training set (streaming mode)
# ----------------------------------------------------------------------------
hf_dataset_train = load_dataset("ILSVRC/imagenet-1k", split="train", streaming=True)
hf_dataset_val = load_dataset("ILSVRC/imagenet-1k", split="validation", streaming=True)


train_stream = hf_dataset_train.take(100_000)
val_stream   = hf_dataset_train.skip(100_000).take(15_000)
test_stream = hf_dataset_val.take(10_000)


# ----------------------------------------------------------------------------
# 2) Preprocessing pipeline
# ----------------------------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def collate_fn(batch):
    """Convert HF samples into tensors"""
    images, labels = [], []
    for sample in batch:
        img = sample["image"]
        if not isinstance(img, Image.Image):  # if numpy array
            img = Image.fromarray(img)
        img = img.convert("RGB")  # <-- ensure 3 channels
        images.append(transform(img))
        labels.append(sample["label"])
    return torch.stack(images), torch.tensor(labels)


# ----------------------------------------------------------------------------
# 3) Wrap datasets with DataLoader
# ----------------------------------------------------------------------------
train_loader = DataLoader(train_stream, batch_size=64, collate_fn=collate_fn)
val_loader   = DataLoader(val_stream,   batch_size=64, collate_fn=collate_fn)
test_loader  = DataLoader(test_stream, batch_size=64, collate_fn=collate_fn)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    top1, top5, total, total_loss = 0, 0, 0, 0.0

    with torch.no_grad():
        for i, (images, labels) in enumerate(dataloader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Top-1
            _, pred1 = outputs.topk(1, dim=1)
            top1 += (pred1.squeeze() == labels).sum().item()

            # Top-5
            _, pred5 = outputs.topk(5, dim=1)
            top5 += sum([labels[j].item() in pred5[j] for j in range(labels.size(0))])

            total += labels.size(0)

            if (i+1) % 1 == 0:
                print(f"[Val Step {i+1}] "
                      f"Loss: {total_loss/(i+1):.4f} | "
                      f"Top-1: {100.*top1/total:.2f}% | "
                      f"Top-5: {100.*top5/total:.2f}%")

    return 100.*top1/total, 100.*top5/total, total_loss/(i+1)

# ----------------------------------------------------------------------------
# 5) Train the model
# ----------------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
custom_model.to(device)

criterion = nn.CrossEntropyLoss()

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

In [151]:
test_top1, test_top5, test_loss = evaluate(
    custom_model, train_loader, criterion, device
)
print(f"\nPTQ Model Results:")
print(f"Top-1: {test_top1:.2f}% | Top-5: {test_top5:.2f}%")

[Val Step 1] Loss: 0.3982 | Top-1: 92.19% | Top-5: 98.44%
[Val Step 2] Loss: 0.5038 | Top-1: 87.50% | Top-5: 97.66%
[Val Step 3] Loss: 0.5387 | Top-1: 86.46% | Top-5: 97.40%
[Val Step 4] Loss: 0.4982 | Top-1: 87.89% | Top-5: 98.05%
[Val Step 5] Loss: 0.5082 | Top-1: 86.88% | Top-5: 97.81%
[Val Step 6] Loss: 0.5080 | Top-1: 86.72% | Top-5: 97.92%
[Val Step 7] Loss: 0.5004 | Top-1: 87.05% | Top-5: 97.99%
[Val Step 8] Loss: 0.5093 | Top-1: 86.91% | Top-5: 97.66%
[Val Step 9] Loss: 0.5186 | Top-1: 86.81% | Top-5: 97.57%


KeyboardInterrupt: 

In [147]:
test_top1, test_top5, test_loss = evaluate(
    custom_model, test_loader, criterion, device
)
print(f"\nPTQ Model Results:")
print(f"Top-1: {test_top1:.2f}% | Top-5: {test_top5:.2f}%")

[Val Step 1] Loss: 1.0826 | Top-1: 73.44% | Top-5: 92.19%
[Val Step 2] Loss: 1.1980 | Top-1: 72.66% | Top-5: 89.84%
[Val Step 3] Loss: 1.1691 | Top-1: 73.44% | Top-5: 90.62%
[Val Step 4] Loss: 1.1930 | Top-1: 72.66% | Top-5: 90.23%
[Val Step 5] Loss: 1.1904 | Top-1: 72.81% | Top-5: 90.31%
[Val Step 6] Loss: 1.2344 | Top-1: 72.40% | Top-5: 89.84%
[Val Step 7] Loss: 1.2653 | Top-1: 71.88% | Top-5: 89.96%
[Val Step 8] Loss: 1.2183 | Top-1: 72.46% | Top-5: 90.23%
[Val Step 9] Loss: 1.2537 | Top-1: 72.05% | Top-5: 89.76%
[Val Step 10] Loss: 1.2720 | Top-1: 71.25% | Top-5: 89.06%
[Val Step 11] Loss: 1.2748 | Top-1: 70.45% | Top-5: 88.92%


KeyboardInterrupt: 

In [152]:
torch.save(custom_model.state_dict(), "BN+softmax+ReLU+fusedBN+foldedBN+fusedReLU+PTQInference7.pth")

drive_path = "/content/drive/MyDrive/BN+softmax+ReLU+fusedBN+foldedBN+fusedReLU+PTQInference7.pth"
shutil.copy("BN+softmax+ReLU+fusedBN+foldedBN+fusedReLU+PTQInference7.pth", drive_path)

'/content/drive/MyDrive/BN+softmax+ReLU+fusedBN+foldedBN+fusedReLU+PTQInference7.pth'

In [153]:
orig_state = torch.load("/content/drive/MyDrive/BN+softmax+ReLU+fusedBN+foldedBN+fusedReLU+PTQInference7.pth")
with open("BN+softmax+ReLU+fusedBN+foldedBN+fusedReLU+PTQInference7.txt", "w") as f:
    for name, param in orig_state.items():
        f.write(f"{name}: {param}\n\n")

drive_path = "/content/drive/MyDrive/BN+softmax+ReLU+fusedBN+foldedBN+fusedReLU+PTQInference7.txt"
shutil.copy("BN+softmax+ReLU+fusedBN+foldedBN+fusedReLU+PTQInference7.txt", drive_path)

'/content/drive/MyDrive/BN+softmax+ReLU+fusedBN+foldedBN+fusedReLU+PTQInference7.txt'